# VolatilityQuant | Feature 4 — Historical Stress Testing

**Purpose:** Compare historical SPY market stress, five-day volatility forecasts, and model errors across the 2008, 2020, and 2022 stress windows. Run cells top to bottom in Google Colab. This notebook is an exploratory historical forecasting backtest, **not** a historical options-hedging backtest.

## 1. Setup and historical prices
Download adjusted SPY closing prices, calculate daily log returns and 20-day rolling annualized volatility.

In [ ]:
!pip install -q yfinance pandas numpy matplotlib arch xgboost

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Use SPY as our market benchmark
ticker = "SPY"

# Download data covering major market crises
start_date = "2005-01-01"
end_date = "2026-10-01"

data = yf.download(
    ticker,
    start=start_date,
    end=end_date,
    auto_adjust=True,
    progress=False
)

if data.empty:
    raise ValueError("No market data downloaded.")

# Handle yfinance column formatting
if isinstance(data.columns, pd.MultiIndex):
    data.columns = data.columns.get_level_values(0)

# Adjusted closing prices
prices = data["Close"].dropna()

# Calculate daily log returns
returns = np.log(
    prices / prices.shift(1)
).dropna()

# Calculate annualized 20-day volatility
volatility = (
    returns.rolling(20).std()
    * np.sqrt(252)
    * 100
)

print("Stock:", ticker)
print("Trading days:", len(prices))
print("Start date:", prices.index.min().date())
print("End date:", prices.index.max().date())

display(data.head())

## 2. Define stress windows and visualize volatility
Highlighted dates are selected research windows, not mechanically detected crises.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# Define historical market stress periods
stress_periods = {
    "2008 Financial Crisis": (
        "2008-09-01", "2009-03-31"
    ),
    "2020 COVID Crash": (
        "2020-02-01", "2020-04-30"
    ),
    "2022 Market Selloff": (
        "2022-01-01", "2022-12-31"
    )
}

# Plot historical volatility
plt.figure(figsize=(14, 6))

plt.plot(
    volatility.index,
    volatility,
    label="20-Day Historical Volatility",
    linewidth=1.2
)

# Highlight stress periods
for name, (start, end) in stress_periods.items():

    plt.axvspan(
        pd.Timestamp(start),
        pd.Timestamp(end),
        alpha=0.20,
        label=name
    )

plt.title("SPY — Historical Volatility During Market Crises")
plt.xlabel("Date")
plt.ylabel("Annualized Volatility (%)")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 3. Market drawdowns
Drawdown is measured against the running peak since the beginning of the full downloaded series, not the peak within each crisis window.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Calculate drawdown from previous historical peaks
running_peak = prices.cummax()

drawdown = (
    prices / running_peak - 1
) * 100

# Compare historical stress periods
stress_results = []

for name, (start, end) in stress_periods.items():

    period_prices = prices.loc[start:end]
    period_vol = volatility.loc[start:end]
    period_drawdown = drawdown.loc[start:end]

    if period_prices.empty:
        continue

    # Return during the selected period
    total_return = (
        period_prices.iloc[-1] /
        period_prices.iloc[0] - 1
    ) * 100

    stress_results.append({
        "Crisis": name,
        "Period Return (%)": total_return,
        "Peak Volatility (%)": period_vol.max(),
        "Average Volatility (%)": period_vol.mean(),
        "Worst Drawdown (%)": period_drawdown.min()
    })

stress_summary = pd.DataFrame(stress_results)

display(stress_summary.round(2))

# Plot historical drawdowns
plt.figure(figsize=(14, 5))

plt.plot(
    drawdown.index,
    drawdown,
    linewidth=1.2
)

for name, (start, end) in stress_periods.items():
    plt.axvspan(
        pd.Timestamp(start),
        pd.Timestamp(end),
        alpha=0.20,
        label=name
    )

plt.title("SPY — Historical Drawdowns")
plt.xlabel("Date")
plt.ylabel("Drawdown from Previous Peak (%)")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Rolling GARCH(1,1) forecasts
Refit GARCH using returns available through each forecast date. Compare the next five trading days’ mean squared returns to expected squared returns. Forecasts at the end of a selected crisis window can have realized outcomes beyond the window.

In [ ]:
from arch import arch_model
import numpy as np
import pandas as pd

# Daily returns in percentage units
returns_pct = returns * 100

HORIZON = 5

garch_stress_results = []

for crisis, (start, end) in stress_periods.items():

    print(f"Evaluating {crisis}...")

    # Forecast dates within the crisis
    crisis_dates = returns_pct.loc[start:end].index

    # Use every fifth trading day
    forecast_dates = crisis_dates[::HORIZON]

    for date in forecast_dates:

        # Use historical information through forecast date
        history = returns_pct.loc[:date]

        if len(history) < 250:
            continue

        # Fit GARCH using past information
        model = arch_model(
            history,
            mean="Constant",
            vol="GARCH",
            p=1,
            q=1,
            dist="t"
        )

        fit = model.fit(
            disp="off",
            show_warning=False
        )

        # Forecast five-day conditional variance
        forecast = fit.forecast(horizon=HORIZON)

        daily_var = (
            forecast.variance.iloc[-1].to_numpy()
        )

        daily_mean = (
            forecast.mean.iloc[-1].to_numpy()
        )

        predicted_var = np.mean(
            daily_var + daily_mean**2
        )

        # Actual next five trading days
        position = returns_pct.index.get_loc(date)

        future = returns_pct.iloc[
            position + 1:position + 1 + HORIZON
        ]

        if len(future) < HORIZON:
            continue

        actual_var = np.mean(
            future.to_numpy()**2
        )

        garch_stress_results.append({
            "Crisis": crisis,
            "Date": date,
            "GARCH_Vol": np.sqrt(
                predicted_var * 252
            ),
            "Actual_Vol": np.sqrt(
                actual_var * 252
            )
        })

garch_stress_df = pd.DataFrame(
    garch_stress_results
)

display(garch_stress_df.head())

print(
    "Total crisis forecasts:",
    len(garch_stress_df)
)

## 5. Pre-crisis XGBoost forecasts
Train a fixed XGBoost model separately before each stress window, purging overlapping five-day targets. GARCH refits within the window, but XGBoost does not; these are **different update policies**. The hyperparameters were selected retrospectively in earlier project work.

In [ ]:
import numpy as np
import pandas as pd
from xgboost import XGBRegressor

HORIZON = 5
EPSILON = 1e-6

# Daily returns in percentage units
r = returns * 100

# Build historical features
features = pd.DataFrame(index=r.index)

for window in [5, 10, 20, 60]:

    features[f"var_{window}d"] = (
        r.pow(2).rolling(window).mean()
    )

    features[f"return_{window}d"] = (
        r.rolling(window).sum()
    )

features["abs_return_1d"] = r.abs()
features["return_1d"] = r
features["negative_return_1d"] = (
    r < 0
).astype(int)

# Future five-day realized variance
future = pd.concat(
    [
        r.shift(-i).pow(2)
        for i in range(1, HORIZON + 1)
    ],
    axis=1
)

features["target"] = future.mean(axis=1)
features.loc[
    features.index[-HORIZON:], "target"
] = np.nan

feature_cols = [
    c for c in features.columns if c != "target"
]

dataset = features.dropna()

xgb_stress_results = []

for crisis, (start, end) in stress_periods.items():

    print(f"Training XGBoost for {crisis}...")

    # Find first crisis observation
    crisis_start = dataset.index.searchsorted(
        pd.Timestamp(start)
    )

    # Purge five-day target overlap
    train = dataset.iloc[
        :max(0, crisis_start - HORIZON)
    ]

    if len(train) < 250:
        print(f"Insufficient training data: {crisis}")
        continue

    # Selected Very Simple XGBoost model
    model = XGBRegressor(
        n_estimators=50,
        max_depth=2,
        learning_rate=0.03,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_lambda=10,
        min_child_weight=10,
        objective="reg:squarederror",
        random_state=42,
        n_jobs=2
    )

    model.fit(
        train[feature_cols],
        np.log(train["target"] + EPSILON)
    )

    # Use the exact same dates as GARCH
    crisis_forecasts = garch_stress_df[
        garch_stress_df["Crisis"] == crisis
    ]

    for _, row in crisis_forecasts.iterrows():

        date = row["Date"]

        X = features.loc[
            [date], feature_cols
        ]

        predicted_log_var = model.predict(X)[0]

        predicted_var = max(
            np.exp(predicted_log_var) - EPSILON,
            1e-8
        )

        xgb_vol = np.sqrt(
            predicted_var * 252
        )

        xgb_stress_results.append({
            "Crisis": crisis,
            "Date": date,
            "XGBoost_Vol": xgb_vol
        })

xgb_stress_df = pd.DataFrame(
    xgb_stress_results
)

# Combine predictions on identical dates
crisis_comparison = garch_stress_df.merge(
    xgb_stress_df,
    on=["Crisis", "Date"],
    how="inner"
)

print(
    "\nForecasts compared:",
    len(crisis_comparison)
)

display(crisis_comparison.head())

## 6. Historical baseline and final accuracy comparison
Evaluate all three methods on identical dates using annualized volatility MAE, RMSE, and signed bias. Lower MAE/RMSE and bias closer to zero are preferred.

In [ ]:
import numpy as np
import pandas as pd

# Add historical baseline on the same forecast dates
crisis_comparison["Baseline_Vol"] = np.nan

for i, row in crisis_comparison.iterrows():

    date = row["Date"]

    # Only returns available through forecast date
    history = returns_pct.loc[:date]

    # Historical mean squared returns
    baseline_var = (
        history.iloc[-20:].pow(2).mean()
    )

    crisis_comparison.loc[i, "Baseline_Vol"] = (
        np.sqrt(baseline_var * 252)
    )

# Compare all three models
final_results = []

for crisis, group in crisis_comparison.groupby("Crisis"):

    actual = group["Actual_Vol"].to_numpy()

    for model in ["Baseline", "GARCH", "XGBoost"]:

        predicted = group[f"{model}_Vol"].to_numpy()

        final_results.append({
            "Crisis": crisis,
            "Model": model,
            "MAE": np.mean(np.abs(predicted - actual)),
            "RMSE": np.sqrt(
                np.mean((predicted - actual)**2)
            ),
            "Bias": np.mean(predicted - actual)
        })

final_crisis_results = pd.DataFrame(final_results)

display(
    final_crisis_results.sort_values(
        ["Crisis", "MAE"]
    ).round(2)
)

## 7. Forecast trajectories
Compare realized volatility against GARCH and XGBoost throughout each stress period.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(
    3, 1,
    figsize=(12, 12)
)

for ax, (crisis, group) in zip(
    axes,
    crisis_comparison.groupby("Crisis", sort=False)
):

    ax.plot(
        group["Date"],
        group["Actual_Vol"],
        label="Actual Volatility",
        color="black",
        linewidth=2
    )

    ax.plot(
        group["Date"],
        group["GARCH_Vol"],
        label="GARCH",
        linewidth=1.8
    )

    ax.plot(
        group["Date"],
        group["XGBoost_Vol"],
        label="XGBoost",
        linewidth=1.8
    )

    ax.set_title(crisis)
    ax.set_ylabel("Annualized Volatility (%)")
    ax.legend()
    ax.grid(alpha=0.3)

plt.xlabel("Forecast Date")
plt.tight_layout()
plt.show()

## 8. Interactive crisis explorer
Choose a stress window to view all three forecasting methods and the corresponding accuracy table.

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.pyplot as plt

# Interactive crisis selection
crisis_selector = widgets.Dropdown(
    options=list(crisis_comparison["Crisis"].unique()),
    description="Crisis:"
)

output = widgets.Output()

def update_crisis(change=None):

    with output:
        clear_output(wait=True)

        selected = crisis_selector.value

        group = crisis_comparison[
            crisis_comparison["Crisis"] == selected
        ].sort_values("Date")

        plt.figure(figsize=(11, 5))

        plt.plot(
            group["Date"],
            group["Actual_Vol"],
            label="Actual Volatility",
            color="black",
            linewidth=2
        )

        for model in ["Baseline", "GARCH", "XGBoost"]:

            plt.plot(
                group["Date"],
                group[f"{model}_Vol"],
                label=model,
                linewidth=1.8
            )

        plt.title(selected + " — Volatility Forecast Comparison")
        plt.xlabel("Date")
        plt.ylabel("Annualized Volatility (%)")
        plt.legend()
        plt.grid(alpha=0.3)
        plt.tight_layout()
        plt.show()

        # Show accuracy for selected crisis
        display(
            final_crisis_results[
                final_crisis_results["Crisis"] == selected
            ].sort_values("MAE").round(2)
        )

crisis_selector.observe(
    update_crisis,
    names="value"
)

display(crisis_selector, output)

update_crisis()

## Methodology and limitations

- **Data:** Adjusted SPY closing prices downloaded from Yahoo Finance for 2005-01-01 to 2026-10-01 (end date exclusive). Downloaded data may change; record the run date.
- **Target:** Next five trading days’ average squared daily log returns (in percent units), displayed as annualized volatility using $\sqrt{252\,\mathrm{mean}(r^2)}$.
- **Historical baseline:** Previous 20 trading days’ mean squared returns.
- **GARCH:** Constant-mean GARCH(1,1) with Student-t innovations, refitted at each forecast date.
- **XGBoost:** Fixed pre-crisis log-variance regression, with features and hyperparameters inherited from Feature 1.
- **Cautions:** This is an exploratory retrospective comparison. The selected model settings reflect later development; model update schedules differ; results cover selected SPY crisis windows only. This notebook does not backtest historical option prices or hedging P&L. Five-day forecast windows are selected approximately nonoverlapping within each crisis.
- 